# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaSabir1/flyrank-ml-internship-laiba_sabir/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib


In [2]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
FACT_QUERY  = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

# Same fix as w03_data_contract.ipynb: two adjacent month partitions so prev30/last30
# sit on either side of a real calendar boundary, not split inside one 31-day partition.
FACT_TWO_MONTHS = (
    f"read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'])"
)
ANCHOR = "DATE '2026-03-01'"
print("Connected. Anchor date:", ANCHOR)

Paste your Hugging Face READ token (hf_...): ··········
Connected. Anchor date: DATE '2026-03-01'


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
print("fact_content_daily_performance columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {FACT_TWO_MONTHS}").df()["column_name"].tolist())
print()
print("dim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").df()["column_name"].tolist())

fact_content_daily_performance columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

dim_content columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized

In [4]:
# All aggregates below use ONLY report_date < ANCHOR (the prev30 window) --
# nothing from the label window (>= ANCHOR) enters this query.
raw = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)                                            AS imp_prev30,
        SUM(f.gsc_clicks)                                                 AS clk_prev30,
        AVG(f.gsc_avg_position)                                           AS pos_prev30,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                             THEN f.report_date END)                      AS days_with_impressions_prev30,
        SUM(CASE WHEN f.ga4_data_available IS TRUE
                  THEN f.ga4_sessions ELSE 0 END)                         AS sessions_prev30,
        SUM(CASE WHEN f.ga4_data_available IS TRUE
                  THEN f.ga4_engaged_sessions ELSE 0 END)                 AS engaged_sessions_prev30,
        BOOL_OR(f.ga4_data_available IS TRUE)                             AS has_ga4_prev30
    FROM {FACT_TWO_MONTHS} f
    WHERE f.report_date < {ANCHOR}
    GROUP BY 1, 2
    HAVING imp_prev30 >= 100          -- same volume floor as ML-04, for a comparable slice
""").df()

content_meta = con.sql(f"""
    SELECT content_hash_id, content_created_date, word_count, content_type, main_intent
    FROM {DIM_CONTENT}
""").df()

df = raw.merge(content_meta, on="content_hash_id", how="left")
print(f"{len(df):,} rows before feature engineering")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

80,322 rows before feature engineering


In [5]:
# --- Engineered numeric features (all knowable before the ANCHOR decision date) ---
df["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(df["content_created_date"])).dt.days
df["ctr_prev30"] = df["clk_prev30"] / df["imp_prev30"].replace(0, np.nan)
df["engagement_rate_prev30"] = df["engaged_sessions_prev30"] / df["sessions_prev30"].replace(0, np.nan)
df["log_impressions_prev30"] = np.log1p(df["imp_prev30"])

# Missingness follows content_type in this data (per data dictionary) -- check before filling,
# then add a has_-flag INSTEAD of blindly fillna(0), so "no keyword data" isn't silently
# encoded as "zero word count" for the model to misread as a real signal.
missing_by_type = df.groupby("content_type", dropna=False)["word_count"].apply(lambda s: s.isna().mean())
print("word_count missing rate by content_type:")
print(missing_by_type.round(3))

df["has_word_count"] = df["word_count"].notna().astype(int)
df["word_count"] = df["word_count"].fillna(0)
df["has_ga4_prev30"] = df["has_ga4_prev30"].fillna(False).astype(int)
df["engagement_rate_prev30"] = df["engagement_rate_prev30"].fillna(0)
df["ctr_prev30"] = df["ctr_prev30"].fillna(0)

# Categorical handling: fillna('unknown') then one-hot -- never leave NaN as a silent category.
for col in ["content_type", "main_intent"]:
    df[col] = df[col].fillna("unknown").astype(str)
cat_encoded = pd.get_dummies(df[["content_type", "main_intent"]], dummy_na=False, dtype=int)

feature_cols_numeric = [
    "imp_prev30", "clk_prev30", "pos_prev30", "days_with_impressions_prev30",
    "sessions_prev30", "engaged_sessions_prev30", "has_ga4_prev30",
    "content_age_days", "word_count", "has_word_count",
    "ctr_prev30", "engagement_rate_prev30", "log_impressions_prev30",
]
feature_frame = pd.concat([df[["client_hash_id", "content_hash_id"] + feature_cols_numeric],
                            cat_encoded], axis=1)
print(f"Feature vector: {feature_frame.shape[0]:,} rows x {feature_frame.shape[1]:,} columns")
feature_frame.head()

word_count missing rate by content_type:
content_type
comparison article    0.000
feedly article        0.116
keyword article       0.314
Name: word_count, dtype: float64
Feature vector: 80,322 rows x 23 columns


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,days_with_impressions_prev30,sessions_prev30,engaged_sessions_prev30,has_ga4_prev30,content_age_days,...,engagement_rate_prev30,log_impressions_prev30,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,client_3ffa76342f366962,content_32bdebcb01540202,551.0,17.0,3.815812,28,0.0,0.0,0,155,...,0.0,6.313548,0,1,0,0,0,0,0,1
1,client_e547b89c05043229,content_d0fa1bbfbc10caf8,957.0,0.0,14.080144,28,10.0,0.0,1,345,...,0.0,6.864848,0,0,1,0,1,0,0,0
2,client_e547b89c05043229,content_4c1e972bec56132e,2882.0,15.0,10.396231,28,6.0,0.0,1,345,...,0.0,7.966587,0,0,1,0,1,0,0,0
3,client_e547b89c05043229,content_64cad58fc02e7605,549.0,0.0,30.741277,28,0.0,0.0,0,345,...,0.0,6.309918,0,0,1,0,1,0,0,0
4,client_e547b89c05043229,content_4e48bd81bb37eb4f,7343.0,3.0,44.960642,28,4.0,0.0,1,345,...,0.0,8.901639,0,0,1,0,1,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [6]:
from IPython.display import Markdown, display

display(Markdown('''
| Feature | Meaning | Missing handling | Available before decision date? |
|---|---|---|---|
| `imp_prev30`, `clk_prev30` | GSC impressions/clicks summed over the 30 days before the anchor | none needed -- volume floor (>=100) already excludes zero-history rows | Yes -- strictly `report_date < ANCHOR` |
| `pos_prev30` | mean GSC avg position, prev30 | none needed | Yes |
| `days_with_impressions_prev30` | how many distinct days in prev30 had any impressions -- a density signal, not just a total | none needed | Yes |
| `sessions_prev30`, `engaged_sessions_prev30` | GA4 sessions/engaged sessions, but ONLY summed where `ga4_data_available IS TRUE` | rows with no/NULL GA4 tracking contribute 0, NOT because engagement was zero -- see `has_ga4_prev30` | Yes, and correctly excludes untracked history instead of treating it as real zero engagement |
| `has_ga4_prev30` | 1 if any prev30 row for this content had real GA4 tracking | boolean OR, defaults False | Yes |
| `content_age_days` | days since `content_created_date` as of the anchor date | none needed (static field, always populated) | Yes -- creation date is always in the past |
| `word_count` + `has_word_count` | article length, with an explicit flag for "not measured" | filled with 0 AND flagged, since missingness follows `content_type` (feedly articles are ~100% missing) -- a blind `fillna(0)` would silently teach the model "content_type == feedly" through a fake zero | Yes -- static content metadata |
| `ctr_prev30`, `engagement_rate_prev30`, `log_impressions_prev30` | derived purely from the prev30 columns above | filled with 0 when the denominator is 0 | Yes -- built only from already-safe prev30 fields |
| `content_type_*`, `main_intent_*` (one-hot) | categorical content metadata | `unknown` bucket before encoding, so a missing category is its own visible column, not silently dropped | Yes -- static metadata |
'''))


| Feature | Meaning | Missing handling | Available before decision date? |
|---|---|---|---|
| `imp_prev30`, `clk_prev30` | GSC impressions/clicks summed over the 30 days before the anchor | none needed -- volume floor (>=100) already excludes zero-history rows | Yes -- strictly `report_date < ANCHOR` |
| `pos_prev30` | mean GSC avg position, prev30 | none needed | Yes |
| `days_with_impressions_prev30` | how many distinct days in prev30 had any impressions -- a density signal, not just a total | none needed | Yes |
| `sessions_prev30`, `engaged_sessions_prev30` | GA4 sessions/engaged sessions, but ONLY summed where `ga4_data_available IS TRUE` | rows with no/NULL GA4 tracking contribute 0, NOT because engagement was zero -- see `has_ga4_prev30` | Yes, and correctly excludes untracked history instead of treating it as real zero engagement |
| `has_ga4_prev30` | 1 if any prev30 row for this content had real GA4 tracking | boolean OR, defaults False | Yes |
| `content_age_days` | days since `content_created_date` as of the anchor date | none needed (static field, always populated) | Yes -- creation date is always in the past |
| `word_count` + `has_word_count` | article length, with an explicit flag for "not measured" | filled with 0 AND flagged, since missingness follows `content_type` (feedly articles are ~100% missing) -- a blind `fillna(0)` would silently teach the model "content_type == feedly" through a fake zero | Yes -- static content metadata |
| `ctr_prev30`, `engagement_rate_prev30`, `log_impressions_prev30` | derived purely from the prev30 columns above | filled with 0 when the denominator is 0 | Yes -- built only from already-safe prev30 fields |
| `content_type_*`, `main_intent_*` (one-hot) | categorical content metadata | `unknown` bucket before encoding, so a missing category is its own visible column, not silently dropped | Yes -- static metadata |


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### 3a. Define the label (from the last30 window only)

In [7]:
label = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) AS imp_last30
    FROM {FACT_TWO_MONTHS}
    WHERE report_date >= {ANCHOR} AND report_date < {ANCHOR} + INTERVAL 30 DAY
    GROUP BY 1
""").df()

data = feature_frame.merge(label, on="content_hash_id", how="inner")
data["is_declining"] = (data["imp_last30"] < 0.8 * data["imp_prev30"]).astype(int)

base_rate = data["is_declining"].mean()
print(f"n = {len(data):,}, base rate (declining) = {base_rate:.3f}")
print("Every score below gets read next to this base rate, not in isolation.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

n = 76,837, base rate (declining) = 0.197
Every score below gets read next to this base rate, not in isolation.


### 3b. Attack #1 — label-derived feature (train with / without the suspect)

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

model_cols = [c for c in feature_frame.columns if c not in ("client_hash_id", "content_hash_id")]

def fit_score(cols, label="run"):
    X = data[cols].fillna(0)
    y = data["is_declining"]
    tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
    tree.fit(X, y)
    auc = roc_auc_score(y, tree.predict_proba(X)[:, 1])
    print(f"{label:40s} ROC AUC: {auc:.3f}  (base rate {base_rate:.3f})")
    return auc

honest_auc = fit_score(model_cols, "HONEST features only")

# Deliberately add the suspect: imp_last30 IS the label's own numerator.
leaky_auc = fit_score(model_cols + ["imp_last30"], "+ imp_last30 (label-derived)")
print(f"\nCollapse from {leaky_auc:.3f} back to {honest_auc:.3f} when removed -- that gap is the confession.")
print("imp_last30 is deleted from the feature set for every step after this cell.")

HONEST features only                     ROC AUC: 0.658  (base rate 0.197)
+ imp_last30 (label-derived)             ROC AUC: 0.855  (base rate 0.197)

Collapse from 0.855 back to 0.658 when removed -- that gap is the confession.
imp_last30 is deleted from the feature set for every step after this cell.


### 3c. Attack #2 — future/overlapping window (`fact_content_query_90d`)

`fact_content_query_90d`'s 90-day window is the most recent ~3 months of the snapshot, which
overlaps my label window. Before touching it as a feature, check whether its window sits
entirely before the anchor.

In [11]:
# fact_content_query_90d has no report_date column -- it's a single fixed 90-day snapshot window,
# not a per-day table, so I can't line it up against ANCHOR the way I can the daily fact table.

print("No report_date column to anchor against, and the docs are explicit that this table's")
print("90-day window overlaps the snapshot's final months. I cannot prove its window sits")
print("entirely before 2026-03-01 -- so any feature built from it goes in Section 4 (excluded),")
print("not into the honest feature set. Demonstrating the risk instead of building the feature:")

qsignal = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(content_visible_query_count) AS visible_queries
    FROM {FACT_QUERY}
    GROUP BY 1
""").df()

risky = data.merge(qsignal, on="content_hash_id", how="left")
risky["visible_queries"] = risky["visible_queries"].fillna(0)
X_risky = risky[model_cols + ["visible_queries"]].fillna(0)
y = risky["is_declining"]
t = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_risky, y)
print(f"\n+ visible_queries (unverified window)   ROC AUC: {roc_auc_score(y, t.predict_proba(X_risky)[:,1]):.3f}")
print("Score barely moves here, which is reassuring, but 'barely moved' is not the same as")
print("'proven safe' -- it stays excluded until I can verify the window against ANCHOR directly.")

No report_date column to anchor against, and the docs are explicit that this table's
90-day window overlaps the snapshot's final months. I cannot prove its window sits
entirely before 2026-03-01 -- so any feature built from it goes in Section 4 (excluded),
not into the honest feature set. Demonstrating the risk instead of building the feature:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


+ visible_queries (unverified window)   ROC AUC: 0.700
Score barely moves here, which is reassuring, but 'barely moved' is not the same as
'proven safe' -- it stays excluded until I can verify the window against ANCHOR directly.


### 3d. Attack #3 — product flags

Confirm none exist in this release to accidentally include.

In [17]:
fact_cols = con.sql(f"DESCRIBE SELECT * FROM {FACT_TWO_MONTHS}").df()["column_name"].tolist()
content_cols = con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").df()["column_name"].tolist()
suspects = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win", "refresh_tier"]

found = [c for c in suspects if c in fact_cols + content_cols]
print("Product-flag columns found in the schema:", found or "none")
print("Confirms the data contract claim from ML-04: these are decision outputs, not shipped here.")

Product-flag columns found in the schema: none
Confirms the data contract claim from ML-04: these are decision outputs, not shipped here.


### 3e. Honest split vs random split — the gap is itself a finding

In [18]:
X_honest = data[model_cols].fillna(0)
y = data["is_declining"]

# Random split (what you'd naively do)
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.2, random_state=42, stratify=y)
random_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_tr, y_tr)
random_auc = roc_auc_score(y_te, random_tree.predict_proba(X_te)[:, 1])

# Grouped split by client_hash_id -- the honest question: does it work on a client it never saw?
splitter = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_honest, y, groups=data["client_hash_id"]))
grouped_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
grouped_tree.fit(X_honest.iloc[train_idx], y.iloc[train_idx])
grouped_auc = roc_auc_score(y.iloc[test_idx], grouped_tree.predict_proba(X_honest.iloc[test_idx])[:, 1])

print(f"Random row split   ROC AUC: {random_auc:.3f}")
print(f"Client-holdout split ROC AUC: {grouped_auc:.3f}")
print(f"Gap: {random_auc - grouped_auc:+.3f}  -- how much of the random-split score was client memorization.")

Random row split   ROC AUC: 0.660
Client-holdout split ROC AUC: 0.506
Gap: +0.153  -- how much of the random-split score was client memorization.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [19]:
display(Markdown('''
| Field | Why excluded |
|---|---|
| `imp_last30` / any last30 GSC metric | Label-derived -- confirmed in Section 3b, it's the numerator my label compares against; using it is circular. |
| `visible_queries` (`fact_content_query_90d`) | Table has no `report_date` to verify its 90-day window sits before the anchor, and the docs say it overlaps recent months -- can't prove it's leakage-free, so it stays out (Section 3c). |
| `health_score`, `priority_score`, `action_type`, other product flags | Confirmed absent from this release's schema (Section 3d) -- would be circular if present. |
| `client_hash_id`, `content_hash_id` | Pseudonymous IDs -- context/grouping only, one-hot encoding them would let the model memorize specific clients/pages instead of generalizing. |
| `main_intent` rows with true nulls beyond "unknown" | Encoded as their own `unknown` category rather than dropped or merged into another bucket, so missingness stays visible instead of silently vanishing. |
'''))


| Field | Why excluded |
|---|---|
| `imp_last30` / any last30 GSC metric | Label-derived -- confirmed in Section 3b, it's the numerator my label compares against; using it is circular. |
| `visible_queries` (`fact_content_query_90d`) | Table has no `report_date` to verify its 90-day window sits before the anchor, and the docs say it overlaps recent months -- can't prove it's leakage-free, so it stays out (Section 3c). |
| `health_score`, `priority_score`, `action_type`, other product flags | Confirmed absent from this release's schema (Section 3d) -- would be circular if present. |
| `client_hash_id`, `content_hash_id` | Pseudonymous IDs -- context/grouping only, one-hot encoding them would let the model memorize specific clients/pages instead of generalizing. |
| `main_intent` rows with true nulls beyond "unknown" | Encoded as their own `unknown` category rather than dropped or merged into another bucket, so missingness stays visible instead of silently vanishing. |


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.